# Demo of bmtool chemical synaptic tuner
By Gregory Glickert


Configure and load cell before using.

In [ ]:
import os, sys, json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from allensdk.api.queries.biophysical_api import BiophysicalApi
from allensdk.model.biophys_sim.config import Config
from allensdk.model.biophysical.utils import Utils

# Lightweight bootstrap for kernels launched outside repo root.
try:
    from modules_local.notebook_helpers import ensure_scp_repo_on_syspath
except ModuleNotFoundError:
    start = Path.cwd().resolve()
    injected = False
    for cand in (start, *start.parents):
        if (cand / "modules_local").is_dir() and (cand / "run_pipeline.py").is_file():
            if str(cand) not in sys.path:
                sys.path.insert(0, str(cand))
            injected = True
            break

    if not injected:
        for base in (start, start.parent):
            try:
                for child in base.iterdir():
                    if child.is_dir() and (child / "modules_local").is_dir() and (child / "run_pipeline.py").is_file():
                        if str(child) not in sys.path:
                            sys.path.insert(0, str(child))
                        injected = True
                        break
                if injected:
                    break
            except Exception:
                pass

    from modules_local.notebook_helpers import ensure_scp_repo_on_syspath

repo_root = ensure_scp_repo_on_syspath()

from modules_local import run_sim


# Local Cell Bundle

### Set Parameters

In [ ]:
cell_name = 'SST' # SST, SST_0, PV

cell_dir = repo_root / 'cells' / cell_name
if not cell_dir.is_dir():
    raise FileNotFoundError(f"Cell directory not found: {cell_dir}")
os.chdir(cell_dir)
print('CWD:', Path.cwd())


model_dir = 'seg_tuned'




In [ ]:
from pathlib import Path
import json

cfg_path = Path('tunes') / model_dir / "cell_configs" / "sim_config.json"
if not cfg_path.is_file():
    cfg_path = Path('tunes') / model_dir / "sim_config.json"
if not cfg_path.is_file():
    cfg_path = repo_root / 'cells' / cell_name / 'tunes' / model_dir / "cell_configs" / "sim_config.json"
if not cfg_path.is_file():
    cfg_path = repo_root / 'cells' / cell_name / 'tunes' / model_dir / "sim_config.json"
if cfg_path.is_file():
    sim_cfg_preview = json.loads(cfg_path.read_text())



### Validate Local Cell Bundle (prepared in Step 0)

In [ ]:
from pathlib import Path

bundle_dir = Path('tunes') / model_dir
required_paths = [
    bundle_dir / "manifest.json",
    bundle_dir / "modfiles",
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    missing_text = "\n".join(missing)
    raise FileNotFoundError(
        f"Missing required local cell bundle files:\n{missing_text}\n"
        "Prepare/download the bundle in Step 0, then rerun this notebook."
    )

print(f"Using existing local model bundle: {bundle_dir.resolve()}")


### Load Precompiled Mechanisms (from Step 0)

In [ ]:
tune_dir = repo_root / 'cells' / cell_name / 'tunes' / model_dir
if not tune_dir.is_dir():
    raise FileNotFoundError(f"Tune directory not found: {tune_dir}")
os.chdir(tune_dir)

from neuron import h
h.load_file("stdrun.hoc")  # Required to use h.run()

mech_candidates = [
    Path("modfiles/x86_64/.libs/libnrnmech.so"),
    Path("modfiles/x86_64/libnrnmech.so"),
]

mech_path = next((p for p in mech_candidates if p.is_file()), None)
if mech_path is None:
    raise FileNotFoundError(
        "Could not find compiled NEURON mechanisms. Expected one of: "
        + ", ".join(str(p) for p in mech_candidates)
        + "\nRun Step 0 to compile modfiles, or use the Colab notebook variant that bootstraps compilation."
    )

h.nrn_load_dll(str(mech_path))
print(f"Loaded mechanisms: {mech_path}")


In [ ]:
################ stim_synapses functions ################
class AllenCell:
    
    ############### Cell Generation Functions ###############
    def __init__(self, gid, soma_diam_multiplier=1.0):
        # from allensdk.model.biophys_sim.config import Config
        # from allensdk.model.biophysical.utils import Utils
        # from neuron import h
        
        self._gid = gid
        self.synapses = []  # Keep track of all synapses
        self.netcons = []   # Keep track of NetCons
        self.stims = []     # Keep VecStims so they don't get garbage-collected
        self.vecs = []
        self.syn_locs = [] 
        
        description = Config().load('manifest.json')

        self.utils = Utils(description)
        self.h = self.utils.h
        self.Vinit = self.utils.description.data['conditions'][0]['v_init']
        # Cast all genome values to float
        for d in self.utils.description.data['genome']:
            if 'value' in d:
                d['value'] = float(d['value'])

        # Load morphology and parameters
        morphology_path = description.manifest.get_path('MORPHOLOGY')
        self.utils.generate_morphology(morphology_path.encode('ascii', 'ignore'))
        self.utils.load_cell_parameters()
        self.setup_morphology(soma_diam_multiplier)
        self._build_section_list()
    
    def setup_morphology(self,soma_diam_multiplier):

        self.soma = self.h.soma
        self.h.soma[0].diam *= soma_diam_multiplier
        self.dend = list(self.h.dend) if hasattr(self.h, 'dend') else []
        self.apic = list(self.h.apic) if hasattr(self.h, 'apic') else []
        self.axon = list(self.h.axon) if hasattr(self.h, 'axon') else []
        from neuron import h

        # Step 1: Set distance origin once (assuming soma(0.5) is your origin)
        h.distance(0, self.soma[0](0.5))

        # Step 2: Classify dendrites into proximal/distal
        self.proximal_dend_segs = []
        self.distal_dend_segs = []

        for sec in self.dend:
            for seg in sec:
                if 20 < h.distance(seg) < 100:
                    self.proximal_dend_segs.append(seg)  
                elif  h.distance(seg) >= 100:
                    self.distal_dend_segs.append(seg)

    def _build_section_list(self):
        # from neuron import h
        self.all = h.SectionList()
        for sec in h.allsec():
            self.all.append(sec)


############### Synapse Generation Functions ###############


    def compute_corrected_weights(self,distances, target_dist_func, bandwidth=10):
        """
        Compute sampling weights that correct for anatomical bias and follow a target distribution.

        Parameters:
        - seg_list: list of (sec, loc, dist)
        - target_dist_func: function mapping distance → desired probability (e.g. scipy poisson.pmf)
        - bandwidth: KDE smoothing bandwidth (in µm)

        Returns:
        - normalized weights (np.array)
        """

        # Step 1: Estimate anatomical density using KDE
        kde = gaussian_kde(distances, bw_method=bandwidth / np.std(distances))
        anatomical_pdf = kde(distances)

        # Step 2: Evaluate target distribution at each distance
        target_pdf = target_dist_func(distances)

        # Step 3: Divide target by anatomical density to correct for bias
        raw_weights = target_pdf / (anatomical_pdf + 1e-12)

        # Step 4: Normalize
        weights = raw_weights / raw_weights.sum()

        return weights
    

    # def _lognormal_mu_sigma(self, mean, std):
    #     """Return μ and σ for np.random.lognormal given arithmetic mean & std."""
    #     mu  = math.log(mean**2 / math.sqrt(std**2 + mean**2))
    #     sig = math.sqrt(math.log(1 + (std**2 / mean**2)))
    #     return mu, sig
    def draw_syn_wt(self, syn_params):
        """Generate a synaptic weight from a lognormal distribution."""


        wt_mean = syn_params.get('wt_mean', syn_params.get('initW', 0.001))
        wt_std  = syn_params.get('wt_std', 0.0) * wt_mean   #scaled off mean, std = 0  → fixed

        if wt_std > 0:
            # mu_logn, sig_logn = self._lognormal_mu_sigma(wt_mean, wt_std)
            mu  = math.log(wt_mean**2 / math.sqrt(wt_std**2 + wt_mean**2))
            sig = math.sqrt(math.log(1 + (wt_std**2 / wt_mean**2)))
            draw_wt = lambda: np.random.lognormal(mu, sig)
        else:
            draw_wt = lambda: wt_mean

        syn_wt = draw_wt()

        return syn_wt
    


    def gen_syn_locs(
            self,
            n_syns,
            dens_eq,        # =lamda d: -0.015*d + 4.25, = lamda: 2.0 
            seg_list,
            ):
        
        """ Generate synapse locations by density distribution dens_eq  """
        
        import math, re, random
        # from neuron import h

        all_syn_locs = []   # Or could use self.syn_locs.append(seg), instead of later when generating syns

        h.distance(0, self.soma[0](0.5))

        ### Generate Synaptic Locations ###
        ## If based on number and probability distrubution
        if n_syns is not None:
            distances = [h.distance(seg) for seg in seg_list]
            weights = self.compute_corrected_weights(distances,dens_eq)

            for syn in range(n_syns):
                sec_seg = random.choices(seg_list, weights=weights, k=1)[0]
                self.syn_locs.append(sec_seg)
                all_syn_locs.append(sec_seg)

        ## If based on density distribution
        else:
            for seg in seg_list:
                seg_dist = h.distance(seg)                  # µm
                seg_len  = seg.sec.L / seg.sec.nseg         # µm

                syn_dens = dens_eq(seg_dist)                # Synapses per µm
                if syn_dens <= 0:                           # skip negative densities
                    print('NEGATIVE/0 SYN_DENS, NO SYN CREATED!!!')
                    continue

                n_syns = math.floor(syn_dens * seg_len)     #+ rng.random())
                if n_syns == 0:
                    print(f'no syns created for seg {seg}: dist {seg_dist}, len {seg_len}, dens {syn_dens}')

                for syn in range(n_syns):
                    self.syn_locs.append(seg)
                    all_syn_locs.append(seg)

                # for _ in range(n_syn):
                #     # random position within the segment’s bounds
                #     #      segment runs from seg.x-Δx to seg.x+Δx
                #     dx = (rng.random() - 0.5) * (1/seg.sec.nseg)
                #     loc = seg.x + dx
                #     syn_locs.append(SynLoc(seg.sec, loc, seg_dist + dx*seg.sec.L))

        return all_syn_locs



    def homogeneous_poisson_timestamps(self, rate_hz, t_start, duration_ms):
        """
        Generate spike times for a homogeneous Poisson process.

        Parameters:
            rate_hz (float): desired firing rate in Hz.
            duration_ms (float): total time to generate over, in milliseconds.

        Returns:
            spike_times (list of float): spike timestamps in ms.
        """
        spike_times = []
        t = t_start         # Start of stimulation
        while t < duration_ms:
            isi = np.random.exponential(1000.0 / rate_hz)  # ISI in ms
            t += isi
            if t < duration_ms:
                spike_times.append(t)
        return spike_times


    def inhomogeneous_poisson_through_num_points(self,lambdas, win_length):
        t = np.zeros(len(lambdas) * win_length)
        lambdas = np.divide(lambdas,1000/win_length)
        
        for i, lambd in enumerate(lambdas):

            num_points = np.random.poisson(lambd)

            if num_points >= win_length:
                t[i * win_length : (i + 1) * win_length] = 1
                continue

            random_inds = np.random.choice(a = np.arange(win_length), size = num_points, replace = False)
            spikes = np.zeros(win_length)
            spikes[random_inds] = 1
            t[i * win_length : (i + 1) * win_length] = spikes

        return t
    

    def gen_spike_times(
            self,
            sim_params,
            syn_params,
            # bio_stim_input,
    ):
        """ Generate spike trains (list of timestamps) for NetCon input. """

        # If constant firing rate, specified in syn_params
        if isinstance(syn_params['freq'], int) or isinstance(syn_params['freq'], float):
            spike_times = self.homogeneous_poisson_timestamps(
                            rate_hz=syn_params['freq'], 
                            t_start=sim_params['tstart'], # 0 if delayed after gen (2 lines down)
                            duration_ms=sim_params['tstop'])
            # spike_times = [spike_time + sim_params['tstart'] for spike_time in spike_times] # Add delay
        
        # If inhomogeneous based on bio data
        elif isinstance(syn_params['freq'], str):

            # file_loc = "../../../external_data/bio_cell_output"
            PFR = pd.read_csv(os.path.join(syn_params['freq']),delimiter=",")
            bio_stim_input = np.array(PFR['AvgFiringRate'][PFR['Time'] >0])

            spike_times = []

            # Generate baseline input until csv data stim
            baseline_factor = bio_stim_input[0]
            baseline_spike_times = self.homogeneous_poisson_timestamps(
                rate_hz=baseline_factor,
                t_start=sim_params['tstart'],
                duration_ms=sim_params['delay']
            )
            for spk in baseline_spike_times:
                spike_times.append(spk)

            # Generate stim input from csv
            # bio_stim_input = [avg_fq - baseline_factor for avg_fq in bio_stim_input]
            stim_spikes = self.inhomogeneous_poisson_through_num_points(
                            bio_stim_input,
                            int(sim_params['bins']))
            time = np.arange(len(stim_spikes))
            stim_spike_times = time[stim_spikes==1]
            stim_spike_times = [spk + sim_params['delay'] for spk in stim_spike_times] # Add delay

            for spk in stim_spike_times:
                spike_times.append(spk)
            
            # Generate baseline input after csv data stim
            # baseline_spike_times = self.homogeneous_poisson_timestamps(
            #     rate_hz=baseline_factor,
            #     t_start=sim_params['tstart'],
            #     duration_ms=sim_params['delay']
            # )
            # for spk in baseline_spike_times:
            #     spike_times.append(spk)


        else:
            print('Please use valid spike train input!')

        return spike_times


    def gen_syn_mechs(
            self,
            syn_loc,
            syn_params,
            ):
        """ Generate syn_name synapse with parameters syn_params. """

        # from neuron import h
        syn_type = syn_params['type']
        # print(f'syn_type: {type(syn_type)} | syn_loc: {syn_loc}')
        syn = getattr(h, syn_params['type'])(syn_loc)
        for param, val in syn_params.items():
            # if param in ('wt_mean', 'wt_std'): # Skip certain keys
            #     continue
            if hasattr(syn, param):
                setattr(syn, param, val)
        syn_wt = self.draw_syn_wt(syn_params)
        syn.initW = syn_wt
        # print(syn.initW)

        return syn



    def gen_syns(
            self,
            syn_params,
            # bio_stim_input,
            sim_params,
            ):
        """ Generate syn_name synapses with parameters syn_params for segments in seg_list. """

        # from neuron import h

        syn_records = {}    # list of dicts we will optionally return
        syn_id = 0

        ### Generate synapses for each syn group in syn_params ###
        for syn_group in syn_params:
            if syn_params[syn_group]['N_syn'] is not None:
                if syn_params[syn_group]['N_syn'] < 1:
                    continue
                
            # Create record lsit for group
            syn_records[syn_group] = []

            # Get syn_params for group
            syn_params_group = syn_params[syn_group]

            # Determine which dendrite sections for group syns
            syn_segs = syn_params_group['segs']
            if syn_segs == 'all':
                seg_list = self.proximal_dend_segs + self.distal_dend_segs
            elif syn_segs == 'proximal':
                seg_list = self.proximal_dend_segs
            elif syn_segs == 'distal':
                seg_list = self.distal_dend_segs
            elif syn_segs == 'soma':
                seg_list = self.soma
            else:
                print('NO DENDRITE SECTIONS SELECTED!!!')


            ### Generate synaptic locations ###
            all_syn_locs = self.gen_syn_locs(
                                n_syns = syn_params_group['N_syn'], 
                                dens_eq = syn_params_group['dist_func'], 
                                seg_list = seg_list,
                                )
            

            for syn_loc in all_syn_locs:
                ### Generate synaptic mechanisms ###
                syn = self.gen_syn_mechs(syn_loc, syn_params_group)
                self.synapses.append(syn)

                ### Generate synaptic input (spike trains) ###
                spike_times = self.gen_spike_times(
                                    sim_params, 
                                    syn_params_group,
                                    )

                ### Generate NetCon & Stimulation ###
                vec = h.Vector(spike_times)
                stim = h.VecStim()
                stim.play(vec)
                nc = h.NetCon(stim, syn)
                nc.weight[0] = 1 
                
                # self.synapses.append(syn)
                self.vecs.append(vec)
                self.stims.append(stim)
                self.netcons.append(nc)


                ### Generate synaptic record ###
                syn_records[syn_group].append({    # Could add more, or auto from syn_params_group/h.
                    "syn_id": syn_id,
                    "group": syn_group,
                    "type": syn_params_group['type'],
                    "weight": syn.initW,
                    "distance": h.distance(syn_loc),
                    "section": syn_loc.sec.name(),
                    "x": syn_loc.x,
                    "spike_times": spike_times,
                    # "NMDA_ratio": nmda_wt, # EXAMPLE, FOR FUTURE
                })
                # print(f'synapse {syn_id} generated: {syn_records[syn_group][-1]}')
                syn_id = syn_id + 1
                
            print(f'{syn_group} synapses generated: {len(all_syn_locs)}')

        return syn_records



    def __str__(self):
        return f"AllenCell(soma={self.soma}, dendrites={len(self.dend)})"



############ Other cell building functions #############


# Build Cell

In [ ]:
from modules_local.notebook_helpers import (
    build_cell_for_notebook,
    resolve_cell_config_for_notebook,
)

def _build_cell(cell_config: dict):
    """Build via shared helpers and expose bmtool-compatible state."""
    loaded = build_cell_for_notebook(cell_config)

    if not hasattr(loaded, 'all'):
        loaded.all = loaded.h.SectionList()
        for sec in loaded.h.allsec():
            loaded.all.append(sec)

    # Keep explicit state containers for notebook-level synapse tests.
    for attr in ('synapses', 'netcons', 'stims', 'vecs', 'syn_locs'):
        if not hasattr(loaded, attr):
            setattr(loaded, attr, [])

    return loaded

def _add_single_synapse(cell, syn_loc, syn_params, spike_train):
    """Notebook-compatible single-synapse attach used in this test cell."""
    from neuron import h

    spec_settings = syn_params.get('spec_settings', {})
    syn_cfg = syn_params.get('spec_syn_param', {})
    mech_name = spec_settings.get('level_of_detail')
    if not mech_name:
        raise KeyError("syn_params['spec_settings']['level_of_detail'] is required")

    syn = getattr(h, mech_name)(syn_loc)
    for param, val in syn_cfg.items():
        if hasattr(syn, param):
            setattr(syn, param, val)

    spike_times = list(spike_train)
    vec = h.Vector(spike_times)
    stim = h.VecStim()
    stim.play(vec)
    nc = h.NetCon(stim, syn)
    nc.weight[0] = 1.0

    cell.synapses.append(syn)
    cell.vecs.append(vec)
    cell.stims.append(stim)
    cell.netcons.append(nc)

cell_config_for_build = resolve_cell_config_for_notebook(cell_name)
cell = _build_cell(cell_config_for_build)



First we must define some general settings and the settings for the connection we would like to tune. Below is an example of what this could look like for excitatory and inhibitory connections. Currently all of these settings, but the ones in the spec_syn_param are needed in order to use the tuner. You can copy all of the general_settings as these should be general enough to use for any case. The spec_settings are going to depend on your exact use case and connection type. 

In [ ]:
syn_tuned = 'PV2SST'

general_settings = {
    'vclamp': True, # if vclamp should start on or off used mostly for singleEventv
    'rise_interval': (0.1, 0.9), #10-90%
    'tstart': 500., # when the singleEvent should start
    'tdur': 100.,    # Dur of sim after single synaptic event has occured
    'threshold': -15., #threshold for spike in mV
    'delay': 1.3, # netcon delay
    'weight': 1., # netcon weight
    'dt': 0.025, # simulation dt
    'celsius': 20 # temp of sim
}

conn_type_settings = {
    'Fac2SST': { # facilitating synapse 
        'spec_settings': {
            'post_cell': 'FSI_Cell', # not used when cell is pre-built
            'vclamp_amp' : -65, #PV/SST Vrest ??????????????????????????????????????? If so -71/-66 respectively
            'sec_x': 0.5, 
            'sec_id': 0, # 1 ???????????????????????????????????????
            "level_of_detail": "AMPA_NMDA_STP",
        },
        'spec_syn_param': {
            'initW': 0.5,
            'tau_r_AMPA': 4,
            'tau_d_AMPA': 5,
            'Use': 0.4,
            'Dep': 0.0,
            'Fac': 70.0,
            'NMDA_ratio': 1.5,
        },
    },
    'Dep2PV': { # depressing synapse
        'spec_settings': {
            'post_cell': 'FSI_Cell',
            'vclamp_amp': -71,
            'sec_x': 0.5,
            'sec_id': 0,
            "level_of_detail": "AMPA_NMDA_STP",  #"GABA_A_STP",
        },
        'spec_syn_param': {
            'initW': 1.5,
            'tau_r_AMPA': 3.5,
            'tau_d_AMPA': 4,
            # 'tau_r_GABAA': 0.9,
            # 'tau_d_GABAA': 15,
            # 'e_GABAA':-75,
            'Use': 0.80,
            'Dep': 100.0,
            'Fac': 0.0,
            'NMDA_ratio': 0.0,
        },
    },
    'Inh2SST': {    # (bg) inhibitory synapse to SST
        'spec_settings': {
            'post_cell': 'FSI_Cell',
            'vclamp_amp': -65, 
            'sec_x': 0.5,
            'sec_id':0,
            "level_of_detail": "GABA_A",  #"GABA_A_STP",
        },
        'spec_syn_param': {
            'initW': 0.1, #0.1 SST/ 2.0 PV #20 
            'tau_r_GABAA': 0.5, #0.9,
            'tau_d_GABAA': 5.5, #15,
            'e_GABAA': -75, #-75
            'gmax': 0.001, #0.001 uS
            # 'Use': 0., #0.4
            # 'Dep': 0., #190
            # 'Fac': 0. #0
        },
    },
    'PV2SST': {    # (bg) inhibitory synapse to SST
        'spec_settings': {
            'post_cell': 'FSI_Cell',
            'vclamp_amp': -65, 
            'sec_x': 0.5,
            'sec_id':0,
            "level_of_detail": "GABA_A_STP",  #"GABA_A_STP",
        },
        'spec_syn_param': {
            'initW': 5.0, #0.1 SST/ 2.0 PV #20 
            'tau_r_GABAA': 0.5, #0.9,
            'tau_d_GABAA': 5.5, #15,
            'e_GABAA': -75, #-75
            'gmax': 0.001, #0.001 uS
            'Use': 1., #0.4
            'Dep': 250., #190
            'Fac': 0. #0
        },
    },

}

In [ ]:
cell = _build_cell(cell_config_for_build)

syn_loc = cell.h.soma[0](0.5)
syn_params = conn_type_settings[syn_tuned]
spike_train = [400,500,600,700,800]

_add_single_synapse(cell, syn_loc, syn_params, spike_train)

sim_params = {
    'dt': 0.025, # ms
    'tstop': 2500.0, # ms
}

sim_traces = run_sim.run_cell(cell,sim_params)


In [ ]:
plt.figure(figsize=(6,2))
plt.plot(sim_traces['T'],sim_traces['V'],color=None)
for spk_time in spike_train:
    plt.axvline(x=spk_time, color='k', linewidth=1)
plt.xlim(300,900)
plt.xlabel('Time (ms)')
plt.ylim(bottom=-68,top=-64.25)
plt.ylabel('Soma Vm (mV)')
plt.title('Membrane Voltage During Synaptic Stimulation')
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(sim_traces['T'],sim_traces['I']*1000,color=None)
for spk_time in spike_train:
    plt.axvline(x=spk_time, color='k', linewidth=1)
plt.xlim(300,900)
plt.xlabel('Time (ms)')
# plt.ylim(bottom=None,top=-64.32)
plt.ylabel('Soma I (pA)')
plt.title('Membrane Voltage During Synaptic Stimulation')
plt.grid()
plt.show()

## initialize tuner
Now we can initialize the synaptic tuner. You must call the SingleEvent method before using the interactiveTuner. When initializing you will have to change a few arguments depending on your use case. other_vars_to_record can be any variable in your synaptic mechanism, while slider_vars can be any range variable you wish to tune in the synapse. If the variable is not defined in the spec_syn_param than the tuner will get the value from the mechanism and try to set up some sliders to tune it.

In [ ]:
cell = _build_cell(cell_config_for_build)


In [ ]:
from pathlib import Path

from modules_local.notebook_helpers import ensure_external_repo_on_syspath

bmtool_path = ensure_external_repo_on_syspath(
    repo_name='bmtool',
    marker_rel=Path('bmtool') / 'synapses.py',
    env_vars=('SCP_BMTOOL_PATH', 'BMTOOL_PATH', 'BMTOOL_ROOT'),
    repo_root=repo_root,
)

from bmtool.synapses import SynapseTuner
from bmtool.synapses import SynapseOptimizer


In [ ]:
# syn_tuned = 'Dep2PV'

# Initialize our tuner
if syn_tuned in ['InhSST', 'Inh2PV',]:
    tuner = SynapseTuner(
        mechanisms_dir='.', # where x86_64 is located
        # mechanisms_dir=mechanisms_dir, # where x86_64 is located
        # templates_dir=templates_file, # where the neuron templates are located

        conn_type_settings=conn_type_settings, # dict of connection settings
        general_settings = general_settings, # dict of general settings

        connection = 'Inh2SST', # key in connection settings for which connection you want to tune
        #json_folder_path=json_folder_path, # If your network uses json files the path can be set to update the connection settings based on the keys and values in the json
        
        current_name = 'i', # name of current variable in synapase
        other_vars_to_record = ['g',], #'gmax_GABAA',], #'v','factor_GABAA'], # Other synaptic variables you wish to record besides the normal ones
        slider_vars=['initW','tau_r_GABAA','tau_d_GABAA','gmax'], # Range variables you want to tune to adjust synaptic response.

        hoc_cell = cell, # cell generated above
    )
elif syn_tuned in ['PV2SST']:
    tuner = SynapseTuner(
        mechanisms_dir='.', # where x86_64 is located
        # mechanisms_dir=mechanisms_dir, # where x86_64 is located
        # templates_dir=templates_file, # where the neuron templates are located

        conn_type_settings=conn_type_settings, # dict of connection settings
        general_settings = general_settings, # dict of general settings

        connection = syn_tuned, # key in connection settings for which connection you want to tune
        #json_folder_path=json_folder_path, # If your network uses json files the path can be set to update the connection settings based on the keys and values in the json
        
        current_name = 'i', # name of current variable in synapase
        other_vars_to_record = ['g',], #'gmax_GABAA',], #'v','factor_GABAA'], # Other synaptic variables you wish to record besides the normal ones
        slider_vars=['initW','Dep','Fac','Use','tau_r_GABAA','tau_d_GABAA','gmax'], # Range variables you want to tune to adjust synaptic response.

        hoc_cell = cell, # cell generated above
    )
elif syn_tuned in ['Fac2SST', 'Dep2PV']:
    tuner = SynapseTuner(
        mechanisms_dir='.', # where x86_64 is located
        # mechanisms_dir=mechanisms_dir, # where x86_64 is located
        # templates_dir=templates_file, # where the neuron templates are located

        conn_type_settings=conn_type_settings, # dict of connection settings
        general_settings = general_settings, # dict of general settings

        connection = syn_tuned, # key in connection settings for which connection you want to tune
        #json_folder_path=json_folder_path, # If your network uses json files the path can be set to update the connection settings based on the keys and values in the json
        
        current_name = 'i', # name of current variable in synapase
        other_vars_to_record = ['record_Pr', 'record_use'], # Other synaptic variables you wish to record besides the normal ones
        slider_vars=['initW','Dep','Fac','Use','tau_r_AMPA','tau_d_AMPA'],

        hoc_cell = cell, # cell generated above
    )

else:
    print('Please choose valid syn from conn_type_settings to tune!!!')

## SingleEvent 
The SingleEvent method will run a short pulse and then print out the synaptic properties for the synapse.  

In [ ]:
tuner.SingleEvent()

## InteractiveTuner
The InteractiveTuner will deliver an input to the cell at a desired weight and frequency. The frequency by default will be 8 spikes then a 250ms delay and then 4 more spikes. 

Paired-pulse ratio is (Avg 2nd pulse - Avg 1st pulse) ÷ 90th percentile amplitude.

Induction is (Avg (6th, 7th, 8th pulses) - Avg 1st pulse) ÷ 90th percentile amplitude. 

Recovery is (Avg (9th, 10th, 11th, 12th pulses) - Avg (1st, 2nd, 3rd, 4th pulses)) ÷ 90th percentile amplitude

In [ ]:
tuner.InteractiveTuner()

In [ ]:
# err

    ## Frequency reponse
We can also see how the STP parameters vary with different train frequencies 

In [ ]:
results = tuner.stp_frequency_response(log_plot=False)

In [ ]:
err

## SynapseOptimizer
If we don't feel like tuner by hand we can also try to optimize an output of our model. In this example we will optimize and find the best STP parameters that give the induction and paired pulse response we want. Something to note is that the optimizer does not know what the trace should look like and only knows the features. So it might get some wild trace that happens to work. Also if you are using the random init_guess and don't like the voltage trace then run it again. The seed is different each time so the optimization will be different and can result in a better fit.

In [ ]:
# Create the optimizer
optimizer = SynapseOptimizer(tuner)

# Define parameter bounds these can be any range variable you wish to tune
param_bounds = {
    'Dep': (0, 200.0),
    'Fac': (0, 400.0),
    'Use': (0.1, 1.0),
    'tau_r_AMPA': (1,4), # tau r needs to be less than tau d so be careful
    'tau_d_AMPA': (5,20)
}

# Define target metrics these are the metrics that the tuner will try to automatic get the synapse to respond with
# max amps is an absolute value
target_metrics = {
    'induction': -0.75,
    'ppr': 0.8,
    'recovery': 0.0,
    'max_amplitude': 25,     # note if you get rid of the max amps in the cost function the fit will normally be better. Then you could scale the max amps with the initW
    'rise_time': 2,          # This wont always be the case, but for this synapse Use controls STP and max amps so it can sometimes struggle to fit.
    'decay_time': 9
}

# currently the only metrics in the SynapseOptimizer are induction, prr, recovery, and max amplitude for train
# for SingleEvent there is just rise and decay time
# error will be constant if not running that case
def custom_cost(metrics, targets):
    # equal zero unless using train input
    induction_error = (metrics['induction'] - targets['induction']) ** 2
    ppr_error = (metrics['ppr'] - targets['ppr']) ** 2
    recovery_error = (metrics['recovery'] - targets['recovery']) ** 2
    max_amp_errror = (metrics['max_amplitude'] - targets['max_amplitude']) ** 2
    # equal zero unless using SingleEvent
    rise_time_error = (metrics['rise_time'] - targets['rise_time']) ** 2
    decay_time_error = (metrics['decay_time'] - targets['decay_time']) ** 2 

    #return rise_time_error + decay_time_error
    return induction_error + 3 * ppr_error + recovery_error + rise_time_error + decay_time_error #+ 0.5*max_amp_errror

# Run optimization with custom cost function
result = optimizer.optimize_parameters(
    target_metrics=target_metrics,
    param_bounds=param_bounds,
    run_single_event=True,  # Run and use parameters from SingleEvent
    run_train_input=True,   # Run and use parameters from train input 
    train_frequency=50,     # Freq in Hz of train input
    train_delay=250,        # delay in ms of second train
    init_guess='random',    # either random or middle_guess. Random will start the synapse witha random value in the param_bound. Middle guess will pick the middle value in the param_bounds
    cost_function=custom_cost,
    method='SLSQP'          # I believe this will be the fastest method, but you may try others check out https://docs.scipy.org/doc/scipy-1.15.0/reference/generated/scipy.optimize.minimize.html
                            # SLSQP is a gradient based method while nelder-mead is simplex (whatever that means)
)

# Plot results
optimizer.plot_optimization_results(result)

## Extra notes
If there is a need for faster optimization with more parameters or more complex metrics the SynapseOptimize could be modifed to use [jax](https://jax.readthedocs.io/en/latest/index.html) and the [scipy method](https://jax.readthedocs.io/en/latest/_autosummary/jax.scipy.optimize.minimize.html). I did not look into this much as my use case currently only takes around 1 min to optimize. Also for the example provided it is important to note that we are using bounded optimizing methods since the Use parameter can not go above one. If there is a case where the parameters have no upper or lower bounds one could look into other optimizing methods